In [3]:
!pip install torch==2.9.0 transformers==4.57.3 pandas==2.2.2 numpy==2.0.2

  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import pandas as pd

In [5]:
# Model Setup
model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=dtype, trust_remote_code=True
).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model Loaded!\n")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model Loaded!



In [8]:
def get_top_n_predictions(logits, n=5):
    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, n)
    return [(tokenizer.decode([int(i)]), float(p)) for i, p in zip(top_ids, top_probs)]

def visualize_step(step, text, preds, chosen):
    print("\n" + "=" * 80)
    print(f"STEP {step}")
    print(f"Current text: \"{text}\"")
    print("\nTop-n-next-token prediction:")

    df = pd.DataFrame({
        "Rank": range(1, 6),
        "Token": [repr(t) for t, _ in preds],
        "Probability": [f"{p:.4f}" for _, p in preds]
    })
    print(df.to_string(index=False))
    print(f"\nSelected token: {repr(chosen)}")

In [ ]:
def generate_step_by_step(
        prompt,
        steps=5
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    current_text = prompt
    print("=" * 80)
    print("AUTOREGRESSIVE GENERATION (GREEDY)")
    print("=" * 80)
    print(f"Prompt: \"{prompt}\"")

    for step in range(steps):
        with torch.no_grad():
            logits = model(input_ids).logits[0, -1]

        next_id = torch.argmax(logits).view(1, 1)
        display_logits = logits

        preds = get_top_n_predictions(display_logits)
        token_str = tokenizer.decode(next_id[0])

        visualize_step(step + 1, current_text, preds, token_str)

        input_ids = torch.cat([input_ids, next_id], dim=-1)
        current_text += token_str

    print("\nFINAL OUTPUT:")
    print(current_text)
    print("=" * 80 + "\n")
